# IBOR → RFR Transition — Difference-in-Differences on Yield-Curve Parameters

Six of the fourteen universe currencies switched their swap curve from an IBOR
reference to a near risk-free rate (RFR) on a known cutover date (see
`data/raw/Zero Rates Data/README.md`). We ask whether that transition shifted
the fitted yield-curve **level** and **slope**
(`src/factors/nonstyle/yield_curve.py`) using a difference-in-differences (DiD)
design. As a robustness check we run the DiD under **two** ways of producing
those parameters — the Nelson-Siegel fit and a simple two-tenor construction
(120M rate as level, 24M minus 120M as slope) — and compare them. Curvature is
not used elsewhere in the project, so it is omitted.

**Variables.** `Treat` = 1 for a currency that transitioned, 0 otherwise;
`Post` = 1 on/after the cutover date, 0 before; `interaction` = `Treat` × `Post`.
The **interaction coefficient is the DiD estimate** — the average change in the
parameter for transitioned currencies around the cutover, net of the
contemporaneous change in the never-transitioned controls.

**Staggered timing.** The cutover dates differ (2022, 2023, 2024), so we split
the treated group into **cohorts** that share a date and estimate one DiD frame
per cohort. Each frame pairs its treated currencies with **all** never-treated
controls over a symmetric window around that cohort's cutover.

**Estimator.** Each frame is estimated with **two-way (entity + time) fixed
effects** via `PanelOLS`. The currency (entity) FE absorbs `Treat` and the date
(time) FE absorbs `Post`, so only `interaction` is passed to `exog`; that single
coefficient is the DiD effect. Estimated separately for level and slope, under
each of the two curve-parameter methods.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from linearmodels.panel import PanelOLS

from src import config
from src.data import loaders
from src.factors.nonstyle.yield_curve import fit_nelson_siegel, fit_simple_level_slope

## 1. Load the zero curve and fit both parameter methods

Each fitter returns one row per `date` × `currency` — already the panel the DiD
needs, no reshaping required. `fit_nelson_siegel` gives model-based `level` /
`slope` (plus an unused `curvature`); `fit_simple_level_slope` gives the raw
two-tenor `level` / `slope`. Both series run continuously across each cutover
(the old IBOR curve feeds them before, the new RFR curve after), so any break at
the cutover is exactly the effect we are after.

In [ ]:
cfg = config.load(PROJECT_ROOT / "config.yaml")
# Anchor the (relative) data root to the project root so the notebook works
# regardless of the directory the kernel is launched from.
cfg.raw["data"]["root"] = str(PROJECT_ROOT / cfg["data"]["root"])

zero_curve = loaders.load_table("zero_curve", cfg)


def to_panel(params):
    """Fitted (date, currency) params -> pandas panel keyed for PanelOLS."""
    p = params.to_pandas()
    p["date"] = pd.to_datetime(p["date"])
    p["currency"] = p["currency"].astype(str)
    return p


# One panel per curve-parameter method, compared head-to-head below.
panels = {
    "nelson_siegel": to_panel(fit_nelson_siegel(zero_curve, cfg)),
    "simple": to_panel(fit_simple_level_slope(zero_curve, cfg)),
}
panels["simple"].head()

## 2. Design: treatment groups, cutover dates, cohorts

Treated currencies and their cutover (first date of the new RFR curve, per the
README) are grouped into cohorts that share a date. Controls are the eight
currencies with a single curve throughout. `WINDOW` is the symmetric event
window around each cutover.

In [ ]:
# Treated: currency -> cutover date (first date of the new RFR-based curve).
TRANSITION_DATES = {
    "GBP": "2022-01-03", "CHF": "2022-01-03", "JPY": "2022-01-03",
    "USD": "2023-07-03", "SGD": "2023-07-03", "CAD": "2024-07-01",
}

# Never-treated controls: a single curve throughout.
CONTROLS = ["AUD", "HKD", "KRW", "TWD", "EUR", "DKK", "NOK", "SEK"]

# Treated currencies sharing a cutover form a cohort, so each DiD frame has one
# unambiguous event date shared by its treated and control units.
COHORTS = {
    pd.Timestamp("2022-01-03"): ["GBP", "CHF", "JPY"],
    pd.Timestamp("2023-07-03"): ["USD", "SGD"],
    pd.Timestamp("2024-07-01"): ["CAD"],
}

WINDOW = pd.DateOffset(months=24)   # symmetric window around each cutover
OUTCOMES = ["level", "slope"]

## 3. Build one DiD frame per method × cohort

Each frame keeps the cohort's treated currencies plus all controls, restricted
to ±`WINDOW` around the cutover. `Treat`, `Post` and `interaction` are attached,
and the frame is indexed by `(currency, date)` — the `(entity, time)` MultiIndex
`PanelOLS` expects. Frames are built for both parameter methods. The balance
check prints observation counts and the pre/post and treated/control shares per
cohort (reported for the Nelson-Siegel panel).

In [ ]:
def build_did_frame(panel, event_date, treated):
    """DiD frame for one cohort: treated + controls within +/- WINDOW of the cutover."""
    keep = set(treated) | set(CONTROLS)
    lo, hi = event_date - WINDOW, event_date + WINDOW
    frame = panel[panel["currency"].isin(keep) & panel["date"].between(lo, hi)].copy()
    frame["treat"] = frame["currency"].isin(treated).astype(int)
    frame["post"] = (frame["date"] >= event_date).astype(int)
    frame["interaction"] = frame["treat"] * frame["post"]
    return frame.set_index(["currency", "date"]).sort_index()


# One set of cohort frames per method: {method: {cohort_date: frame}}.
did_frames = {
    method: {d: build_did_frame(p, d, t) for d, t in COHORTS.items()}
    for method, p in panels.items()
}

for d, f in did_frames["nelson_siegel"].items():
    n_ent = f.index.get_level_values("currency").nunique()
    print(f"{d.date()}: {f.shape[0]} rows, {n_ent} currencies, "
          f"post share {f['post'].mean():.2f}, treated share {f['treat'].mean():.2f}")

## 4. Two-way fixed-effects DiD

For each method, cohort frame and outcome:

$$y_{it} = \alpha_i + \gamma_t + \delta\,(\text{Treat}_i \times \text{Post}_{it}) + \varepsilon_{it}$$

with currency fixed effects $\alpha_i$ (`entity_effects`) and date fixed effects
$\gamma_t$ (`time_effects`). The entity FE absorbs the time-invariant `Treat` and
the time FE absorbs the entity-invariant `Post`, so both are omitted from `exog`
and only `interaction` is estimated — its coefficient $\delta$ is the DiD effect.
Standard errors are clustered by currency.

> **Caveat:** each frame has only ~9–11 currency clusters, well below the usual
> threshold for reliable cluster-robust inference; read the p-values as
> indicative rather than exact.

In [ ]:
def run_twfe(frame, outcome, treated_currencies):
    """TWFE DiD with Heterogeneous Treatment Effects per currency."""
    # 1. Create entity-specific interaction terms (Post * Entity_Dummy)
    exog_cols = []
    for cur in treated_currencies:
        term_name = f"interaction_{cur}"
        # frame is multi-indexed by ["currency", "date"]; grab the currency level
        is_cur = (frame.index.get_level_values("currency") == cur).astype(int)
        frame[term_name] = is_cur * frame["post"]
        exog_cols.append(term_name)
    
    # 2. Run the panel regression with multiple interaction terms
    mod = PanelOLS(
        frame[outcome],
        frame[exog_cols],
        entity_effects=True,
        time_effects=True,
    )
    return mod.fit(cov_type="clustered", cluster_entity=True)


records = []
for method, frames in did_frames.items():
    for d, f in frames.items():
        treated_currencies = COHORTS[d] # e.g., ["USD", "SGD"]
        for y in OUTCOMES:
            res = run_twfe(f, y, treated_currencies)
            
            # 3. Extract and record the unique effect for EACH treated currency
            for cur in treated_currencies:
                term = f"interaction_{cur}"
                records.append({
                    "method": method,
                    "cohort": d.date(),
                    "currency": cur,      # Replaced the pooled 'treated' string
                    "outcome": y,
                    "did_coef": res.params[term],
                    "std_error": res.std_errors[term],
                    "t_stat": res.tstats[term],
                    "p_value": res.pvalues[term],
                    "n_obs": int(res.nobs),
                })

results = pd.DataFrame.from_records(records)
results

## 5. Results — Nelson-Siegel vs simple

`did_coef` is the estimated shift in each parameter (in the same decimal units as
`zero_rate`) attributable to the IBOR→RFR transition for that cohort. The pivot
below puts the two methods side by side for each cohort × outcome, so a robust
effect is one that agrees in sign and rough magnitude across methods.

In [ ]:
results.pivot_table(
    index=["cohort", "currency", "outcome"], columns="method", values="did_coef"
)

In [ ]:
import plotnine as p9

# Toggle this between 'nelson_siegel' and 'simple' as needed
target_method = "simple" 
outcome = "level"

df_plot = results[
    (results["method"] == target_method) & 
    (results["outcome"] == outcome)
].copy()

# Sort the dataframe by the coefficient to properly order the x-axis
df_plot = df_plot.sort_values("did_coef")

# Convert 'currency' to an ordered Categorical type based on the sorted order
# This forces plotnine to respect the most-negative to most-positive order
df_plot["currency"] = pd.Categorical(
    df_plot["currency"], 
    categories=df_plot["currency"], 
    ordered=True
)

# Build the plotnine figure
fig = (
    p9.ggplot(df_plot, p9.aes(x="currency", y="did_coef"))
    + p9.geom_hline(yintercept=0, linetype="dashed", color="red", alpha=0.7)
    + p9.geom_errorbar(
        p9.aes(ymin="did_coef - std_error", ymax="did_coef + std_error"), 
        width=0.2, 
        color="black"
    )
    + p9.geom_point(color="blue", size=3)
    + p9.labs(
        title=f"DiD Impact on Yield Curve {outcome.capitalize()} ({target_method})",
        x="Currency",
        y="Estimate (± 1 Std Error)"
    )
    + p9.theme_minimal()
    + p9.theme(
        figure_size=(6, 4),
        plot_title=p9.element_text(weight="bold")
    )
)

# Display the plot
fig

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def run_event_study(frame, event_date, outcome, treated_currencies):
    """
    Runs a dynamic DiD (Event Study) to test parallel trends.
    """
    df = frame.copy().reset_index()
    
    # 1. Calculate relative time in months (approx 30 days)
    # E.g., -24, -23 ... -1, 0, 1 ... 24
    df['days_to_event'] = (df['date'] - pd.to_datetime(event_date)).dt.days
    df['rel_month'] = np.floor(df['days_to_event'] / 30).astype(int)
    
    # 2. Treat indicator (1 if in treated group, 0 otherwise)
    df['is_treated'] = df['currency'].isin(treated_currencies).astype(int)
    
    # 3. Create interaction dummies for each relative month
    # We omit relative month -1 as the baseline/reference period
    unique_months = sorted(df['rel_month'].unique())
    reference_month = -1
    
    interaction_cols = []
    for m in unique_months:
        if m == reference_month:
            continue # Skip reference to avoid perfect multicollinearity
            
        col_name = f"lag_{m}" if m < 0 else f"lead_{m}"
        df[col_name] = ((df['rel_month'] == m) & (df['is_treated'] == 1)).astype(int)
        interaction_cols.append(col_name)
        
    # Re-index for PanelOLS (currency, date)
    df = df.set_index(['currency', 'date'])
    
    # 4. Run TWFE regression
    mod = PanelOLS(
        df[outcome],
        df[interaction_cols],
        entity_effects=True,
        time_effects=True,
    )
    res = mod.fit(cov_type="clustered", cluster_entity=True)
    
    return res, unique_months, reference_month

In [ ]:
def plot_event_study(res, unique_months, reference_month, outcome, title):
    """Plots the event study coefficients to visually test parallel trends."""
    coefs = []
    cis_lower = []
    cis_upper = []
    plot_months = []
    
    for m in unique_months:
        plot_months.append(m)
        if m == reference_month:
            # Reference month is exactly 0 by definition
            coefs.append(0)
            cis_lower.append(0)
            cis_upper.append(0)
        else:
            col_name = f"lag_{m}" if m < 0 else f"lead_{m}"
            coefs.append(res.params[col_name])
            # 95% Confidence Interval (approx +/- 1.96 * std_error)
            se = res.std_errors[col_name]
            cis_lower.append(res.params[col_name] - 1.96 * se)
            cis_upper.append(res.params[col_name] + 1.96 * se)
            
    plt.figure(figsize=(10, 5))
    plt.axhline(0, color='black', linewidth=1, linestyle='--')
    plt.axvline(0, color='red', linewidth=1, linestyle='-', label='Cutover (Transition)')
    
    plt.errorbar(
        plot_months, coefs, 
        yerr=[np.array(coefs) - np.array(cis_lower), np.array(cis_upper) - np.array(coefs)],
        fmt='o', color='blue', ecolor='lightblue', elinewidth=3, capsize=0
    )
    
    plt.title(f"Event Study for {title} (Outcome: {outcome})")
    plt.xlabel("Months relative to cutover")
    plt.ylabel("DiD Estimate (Difference in Yield Curve Parameter)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# Example execution for the USD/SGD cohort on the Nelson-Siegel Level parameter
f = did_frames["nelson_siegel"][pd.Timestamp("2023-07-03")]
res, mths, ref = run_event_study(f, "2023-07-03", "level", ["USD", "SGD"])
plot_event_study(res, mths, ref, "level", "2023 Cohort (USD, SGD)")

In [ ]:
import pandas as pd
import numpy as np
from plotnine import *

# 1. Prepare data in Relative Event Time (using Nelson-Siegel as an example)
frames_list = []
for event_date, frame in did_frames["nelson_siegel"].items():
    df = frame.reset_index().copy()
    # Calculate relative days, then bin into ~30-day "months"
    df["relative_days"] = (df["date"] - pd.Timestamp(event_date)).dt.days
    df["relative_month"] = (df["relative_days"] // 30)
    frames_list.append(df)

event_df = pd.concat(frames_list, ignore_index=True)

# 2. Calculate averages for Treated vs Control by relative month
trend_data = event_df.groupby(["treat", "relative_month"])[["level", "slope"]].mean().reset_index()
trend_data["Group"] = trend_data["treat"].map({1: "Treated", 0: "Control"})

# 3. Plotnine Graph for the 'Level' parameter
p_trends = (
    ggplot(trend_data, aes(x="relative_month", y="level", color="Group"))
    + geom_line(size=1)
    + geom_vline(xintercept=0, linetype="dashed", color="red", size=1)
    + annotate("text", x=1, y=trend_data["level"].max(), label="Transition Date", color="red", ha="left")
    + scale_color_manual(values={"Treated": "#1f77b4", "Control": "#ff7f0e"})
    + theme_minimal()
    + labs(
        title="Parallel Trends Check: Mean NS Level by Relative Time",
        x="Months Relative to IBOR->RFR Transition",
        y="Average Yield Curve Level",
        color=""
    )
    + theme(figure_size=(8, 4))
)
p_trends

## Caveats

* **Few clusters.** Each cohort has ~9–11 currencies; cluster-robust standard
  errors are approximate at this cluster count.
* **Parallel trends.** DiD identification assumes treated and control curves
  would have moved together absent the transition. This is not tested here (an
  event-study on relative-time leads/lags is the natural next step).
* **Mechanical repricing.** The reference rate itself changes at the cutover
  (IBOR carries credit/term premia over a near-RFR), so a nonzero DiD partly
  reflects that repricing, not only curve-shape dynamics.
* **Fixed `tau`.** The NS decay is held fixed; the CAD post-window is truncated
  by the data end (2026-06-11).